# Model vs students on unseen (`unknown`) events

Both prediction sources score ~98% on the trigger-word-labeled rows — but that is
mostly leakage: those rows are (near-)training data and every one contains the keyword
the model learned. The only honest test is the `unknown` events: no trigger word,
never trained on. There is no ground truth there, so we hand-labeled two samples from
the model's 20-class taxonomy:

- **`random_unknown_labeled.csv`** — 80 *randomly* drawn `unknown` events (unbiased).
- **`manual_labeled_unknown.csv`** — 70 `unknown` events where the two models *disagree*.

The story below: the honest accuracy, why the sampling choice nearly tripled the
apparent gap, and where the real gap comes from.

In [1]:
import pandas as pd
import plotly.express as px

def score(path):
    d = pd.read_csv(path)
    d['model_correct'] = d['predicted_class_model'] == d['manual_label']
    d['students_correct'] = d['predicted_class_students'] == d['manual_label']
    d['agree'] = d['predicted_class_model'] == d['predicted_class_students']
    return d

rand = score('random_unknown_labeled.csv')
disag = score('manual_labeled_unknown.csv')
print(f'random sample: {len(rand)} events | disagreement sample: {len(disag)} events')

random sample: 80 events | disagreement sample: 70 events


## 1. The honest number

On a random sample of `unknown` events the model gets **~39%** right and the students'
model **~50%**. Both are mediocre on truly unseen text — nowhere near the 98% on
labeled data — and the real gap is about **11 points**, not the 3–4x it can look like.

In [2]:
n = len(rand)
ma, sa = rand['model_correct'].mean(), rand['students_correct'].mean()
ci = lambda p: 1.96 * (p * (1 - p) / n) ** 0.5  # normal-approx 95% CI (small n!)

head = pd.DataFrame({'model': ['Model (bert, 4 ep)', 'Students (t5, 6 ep)'],
                     'accuracy': [ma, sa], 'ci': [ci(ma), ci(sa)]})
fig = px.bar(head, x='model', y='accuracy', color='model', text=head['accuracy'].round(3),
             error_y=head['ci'], labels={'model': '', 'accuracy': 'Accuracy on random unknown'})
fig.update_layout(title={'text': f'Honest accuracy on random unknown events (n={n}, 95% CI)', 'x': 0.5},
                  plot_bgcolor='white', width=620, height=430, yaxis={'range': [0, 1]}, showlegend=False)
fig.show()

## 2. Why the sample choice matters

Score the models **only where they disagree** and the model craters to ~10% while the
students barely move — a fake 4x gap. Disagreement-only sampling over-selects each
model's odd predictions, and the smaller model emits more of them, so it gets punished
twice. Sampling randomly shrinks the gap to ~11 points.

In [3]:
comp = pd.DataFrame({
    'model': ['Model', 'Students', 'Model', 'Students'],
    'sample': ['Random (honest)', 'Random (honest)',
               'Disagreement-only (biased)', 'Disagreement-only (biased)'],
    'accuracy': [rand['model_correct'].mean(), rand['students_correct'].mean(),
                 disag['model_correct'].mean(), disag['students_correct'].mean()],
})
fig = px.bar(comp, x='model', y='accuracy', color='sample', barmode='group',
             text=comp['accuracy'].round(3), labels={'model': '', 'accuracy': 'Accuracy'})
fig.update_layout(title={'text': 'Random vs disagreement-only sampling', 'x': 0.5},
                  plot_bgcolor='white', width=700, height=430, yaxis={'range': [0, 1]},
                  legend={'title': ''})
fig.show()

## 3. Where the gap lives

On the random sample the models **agree only 40%** of the time, and that shared guess
is right ~62%. On the 60% where they **disagree**, the students' model wins about 2:1 —
but **neither is right a third of the time**: the genuinely ambiguous `unknown` events.

In [4]:
ag, dis = rand[rand['agree']], rand[~rand['agree']]
print(f'agree on {len(ag)}/{len(rand)} ({len(ag)/len(rand):.0%}); shared label right {ag["model_correct"].mean():.0%}')

brk = pd.DataFrame({
    'outcome': ['Model right', 'Students right', 'Neither right'],
    'share': [dis['model_correct'].mean(), dis['students_correct'].mean(),
              (~dis['model_correct'] & ~dis['students_correct']).mean()],
})
fig = px.bar(brk, x='outcome', y='share', color='outcome', text=brk['share'].round(3),
             labels={'outcome': '', 'share': 'Share of disagreements'})
fig.update_layout(title={'text': f'When the models disagree (n={len(dis)})', 'x': 0.5},
                  plot_bgcolor='white', width=620, height=430, yaxis={'range': [0, 1]}, showlegend=False)
fig.show()

agree on 32/80 (40%); shared label right 62%


## Takeaway

- **Real unknown accuracy: ~39% (model) vs ~50% (students).** The students' t5 model
  generalizes better, but both are weak on unseen text.
- The **98% labeled accuracy is leakage**, not skill — only the `unknown` set is honest.
- **Always measure on a random sample.** Disagreement-only sampling inflated the gap ~3x.
- Small n (80, ±~11pp): the direction is solid, the exact gap is fuzzy.